In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
mql = pd.read_csv("../data/raw/olist_marketing_qualified_leads_dataset.csv")

In [9]:
# First rows
mql.head()

,mql_id,first_contact_date,landing_page_id,origin
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search


In [10]:
# Number of rows and columns
print("Shape:", mql.shape)

# Column names
print("Columns:", mql.columns.tolist())

# Data types and non-null counts
mql.info()

# Statistical summary
mql.describe()

Shape: (8000, 4)
Columns: ['mql_id', 'first_contact_date', 'landing_page_id', 'origin']
<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   mql_id              8000 non-null   str  
 1   first_contact_date  8000 non-null   str  
 2   landing_page_id     8000 non-null   str  
 3   origin              7940 non-null   str  
dtypes: str(4)
memory usage: 250.1 KB


,mql_id,first_contact_date,landing_page_id,origin
count,8000,8000,8000,7940
unique,8000,336,495,10
top,dac32acd4db4c29c230538b72f8dd87d,2018-05-02,b76ef37428e6799c421989521c0e5077,organic_search
freq,1,93,912,2296


In [11]:

# 1. Check missing values
print("Missing values:")
print(mql.isnull().sum())

# 2. Check duplicate rows
print("\nFully duplicated rows:", mql.duplicated().sum())

# 3. Check duplicate MQL IDs
print("Duplicate MQL IDs:", mql["mql_id"].duplicated().sum())

# 4. Check marketing origins
print("\nMarketing origin counts:")
print(mql["origin"].value_counts(dropna=False))

# 5. Check landing page IDs
print("\nUnique landing pages:", mql["landing_page_id"].nunique())

# 6. Check date format
print("\nFirst contact date data type:")
print(mql["first_contact_date"].dtype)

print("\nDate range:")
print(mql["first_contact_date"].min())
print(mql["first_contact_date"].max())

# 7. Check blank values in text columns
text_cols = ["mql_id", "landing_page_id", "origin"]

for col in text_cols:
    print(f"\nBlank values in {col}:",
          mql[col].dropna().str.strip().eq("").sum())


Missing values:
mql_id                 0
first_contact_date     0
landing_page_id        0
origin                60
dtype: int64

Fully duplicated rows: 0
Duplicate MQL IDs: 0

Marketing origin counts:
origin
organic_search       2296
paid_search          1586
social               1350
unknown              1099
direct_traffic        499
email                 493
referral              284
other                 150
display               118
other_publicities      65
NaN                    60
Name: count, dtype: int64

Unique landing pages: 495

First contact date data type:
str

Date range:
2017-06-14
2018-05-31

Blank values in mql_id: 0

Blank values in landing_page_id: 0

Blank values in origin: 0


In [12]:

# Create a copy for cleaning
mql_cleaned = mql.copy()

# 1. Convert first contact date to datetime
mql_cleaned["first_contact_date"] = pd.to_datetime(
    mql_cleaned["first_contact_date"],
    errors="coerce"
)

# 2. Replace missing origins with "unknown"
mql_cleaned["origin"] = mql_cleaned["origin"].fillna("unknown")

# 3. Remove unnecessary leading and trailing spaces
mql_cleaned["origin"] = mql_cleaned["origin"].str.strip()

# 4. Check the results
print("Original shape:", mql.shape)
print("Cleaned shape:", mql_cleaned.shape)

print("\nUpdated data types:")
print(mql_cleaned.dtypes)

print("\nMissing values:")
print(mql_cleaned.isnull().sum())

print("\nMarketing origin counts:")
print(mql_cleaned["origin"].value_counts())

print("\nDuplicate MQL IDs:",
      mql_cleaned["mql_id"].duplicated().sum())

print("\nDate range:")
print(mql_cleaned["first_contact_date"].min())
print(mql_cleaned["first_contact_date"].max())

print("\nPreview:")
print(mql_cleaned.head())


Original shape: (8000, 4)
Cleaned shape: (8000, 4)

Updated data types:
mql_id                           str
first_contact_date    datetime64[us]
landing_page_id                  str
origin                           str
dtype: object

Missing values:
mql_id                0
first_contact_date    0
landing_page_id       0
origin                0
dtype: int64

Marketing origin counts:
origin
organic_search       2296
paid_search          1586
social               1350
unknown              1159
direct_traffic        499
email                 493
referral              284
other                 150
display               118
other_publicities      65
Name: count, dtype: int64

Duplicate MQL IDs: 0

Date range:
2017-06-14 00:00:00
2018-05-31 00:00:00

Preview:
                             mql_id first_contact_date  \
0  dac32acd4db4c29c230538b72f8dd87d         2018-02-01   
1  8c18d1de7f67e60dbd64e3c07d7e9d5d         2017-10-20   
2  b4bc852d233dfefc5131f593b538befa         2018-03-22   
3  6

In [13]:
mql_cleaned.to_csv("mql_cleaned.csv", index=False)

print("MQL dataset saved successfully!")

MQL dataset saved successfully!


Marketing Qualified Leads (MQL) Dataset – Data Cleaning Summary

The Olist MQL dataset originally contained 8,000 rows and 4 columns.

Cleaning Steps

Checked for missing values; 60 missing values were found in the origin column.
Checked for duplicate rows and duplicate mql_id values; none were found.
Replaced the 60 missing origin values with "unknown", preserving all records.
Retained the existing 1,099 "unknown" origin entries.
Converted first_contact_date from string to datetime format.
Checked landing_page_id for missing values and duplicates; no changes were necessary.
Removed unnecessary leading and trailing spaces from the origin column.
Final Result

The cleaned dataset retains all 8,000 rows and 4 columns. The origin column now contains 1,159 "unknown" entries, and the contact dates are stored in datetime format. The cleaned version is saved as mql_cleaned.csv, while the original dataset remains unchanged.

Relevance to Research

This dataset is essential for Research Question 5, as it provides information about potential sellers and their marketing acquisition channels. It will be used to investigate differences in seller lead conversion across marketing origins and will potentially support Research Question 6 when connected to the closed deals and seller datasets.